In [1]:
perfil = "SRD1CIV"
tipo_processo = "EXECUÇÃO FISCAL"

In [2]:
# Importa tudo, loga, entra no perfil

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import ElementClickInterceptedException

#Bibliotecas de Sistema
from datetime import datetime
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false
from pydoc import text
from sympy import true

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Downloads"

navegador = eproc.novo_browser(pasta_downloads)

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

#eproc.login_no_eproc_tj(navegador, username, password, pyotop_code)
eproc.login_no_eproc(navegador, username, password, pyotop_code)

# Entra no perfil da Vara
eproc.entrar_no_perfil(navegador, perfil)

Driver do Eproc importado
Perfil carregado: SRD1CIV


In [3]:
#Define funções para pegar os dados da tabela

def pega_tabela_pagina(dados_tabela):
    tabela = navegador.find_element(By.ID, "tabelaLocalizadores")
    linhas = tabela.find_elements(By.TAG_NAME, "tr")[1:]  # Ignora o cabeçalho
    lastpage = false
    while lastpage == false:
        for linha in linhas:
            # Aguarda o carregamento do tbody da tabela antes de processar as linhas
            WebDriverWait(navegador, 10).until(
                EC.presence_of_element_located((By.XPATH, "//table[@id='tabelaLocalizadores']/tbody"))
            )
            colunas = linha.find_elements(By.TAG_NAME, "td")[:4]
            if len(colunas) >= 3:
                # Adiciona apenas a primeira linha, sem quebra de linha
                dados_tabela.append([colunas[1].text.split('\n')[0], colunas[2].text.split('\n')[0], colunas[3].text.split('\n')[0]])
            
            lastpage = true

def pega_ultima_peticao(navegador):
    documentos_eventos = []
    eventos = navegador.find_elements(By.CLASS_NAME, "td-evento")

    for evento in eventos:
        doc_id = evento.get_dom_attribute("id")
        tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
        evento_id = tr_element.find_element(By.XPATH, './td[2]').text
        # Garante que evento_id seja apenas um int (remove qualquer caractere não numérico)
        evento_id = ''.join(filter(str.isdigit, evento_id))
        
        # Pega o atributo data-nome do elemento com classe infraLinkDocumento dentro do link
        try:
            infra_link = evento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
            data_nome = infra_link.get_attribute("data-nome")
        except Exception:
            data_nome = None

        documentos_eventos.append((doc_id, data_nome, evento_id))

    documentos_eventos_filtrados = [item for item in documentos_eventos if item[1] == "PET"]

    documento_requerido = documentos_eventos_filtrados[0]
    texto_peticao = eproc.pega_texto_documento(navegador, documento_requerido[0])

    return evento_id, texto_peticao


def pega_texto_documento(navegador, documento):
    WebDriverWait(navegador, 20).until(
        EC.presence_of_element_located((By.ID, documento))
    )
    # 1. Localizar o elemento pelo ID
    elemento = navegador.find_element(By.ID, documento)
    # 2. Criar ActionChains para executar o mouse over
    actions = ActionChains(navegador)
    # Rolar a página para o elemento antes de mover o mouse
    navegador.execute_script("arguments[0].scrollIntoView(true); window.scrollBy(0, -150);", elemento)
    # Faz o mouseover em cima do texto link infraLinkDocumento do elemento
    link_doc = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
    actions.move_to_element(link_doc).perform()
    # 3. Aguardar para o hover ter efeito
    time.sleep(5)
    
    # Verifica se há uma div com a classe 'divBoxPreview' visível na página
    overlays = navegador.find_elements(By.ID, "divBoxPreview")
    visiveis = [div for div in overlays if div.is_displayed()]

    conteudo = ""
    if visiveis:
        div = overlays[0]
        # Move o foco para a div
        ActionChains(navegador).move_to_element(div).click().perform()
        # Aguarda carregar o conteúdo (ajuste o tempo se necessário)
        time.sleep(1)
        # Seleciona o texto
        conteudo = navegador.find_element(By.ID, "divBoxPreview").text
        # Clica no botão de fechar o preview, se existir
        btn_close = navegador.find_element(By.ID, "divClosePreview")
        btn_close.click()    
        time.sleep(3)

    else:
        print("Erro ao recuperar o documento")
    return conteudo


def ollama_resumo(pedido):
    print("========== Iniciando resumo com LLM... ==========")
    pergunta_gemma = "Considere o seguinte pedido." \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. " \
    "O resumo deve ser genérico e breve (uma frase apenas, com o mínimo de palavras possível). " \
    "Se tiver mais de um pedido, retorne uma frase para cada um." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])

def verifica_tipos_de_pedidos(pedido, lista_de_pedidos):
    print("========== Verificando se é um caso de uso conhecido... ==========")
    pergunta_gemma = "Considere a seguinte lista de pedidos:" \
    f"{lista_de_pedidos}" \
    f"É possível dizer que o pedido '{pedido}' pode ser adequadamente descrito por um item dessa lista?." \
    "Se sim, retorne APENAS o texto EXATO do resumo do pedido correspondente na lista. Se não, retorne APENAS o texto 'Não'."

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])




In [ ]:
#TESTE
# Verifica se há uma div com a classe 'divBoxPreview' visível na página
overlays = navegador.find_elements(By.ID, "divBoxPreview")
visiveis = [div for div in overlays if div.is_displayed()]

conteudo = ""
if visiveis:
    div = overlays[0]
    # Move o foco para a div
    ActionChains(navegador).move_to_element(div).click().perform()
    # Aguarda carregar o conteúdo (ajuste o tempo se necessário)
    # Seleciona o texto
    navegador.switch_to.frame("fraBoxPreview")
    # Pega todo o texto do que está em #document dentro do iframe
    iframe = navegador.find_element(By.ID, "conteudoIframe")
    conteudo = navegador.execute_script("return arguments[0].contentDocument.documentElement.innerText;", iframe)
    navegador.switch_to.default_content()
    # Clica no botão de fechar o preview, se existir 

print("Conteúdo do documento:", conteudo)



Conteúdo do documento: 


In [ ]:
#Pega os processos novos para minutar, adiciona coluna de dias, pagina por 100

# Espera até o elemento estar presente e clicável
meus_localizadores = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'i[title="Meus Localizadores"]'))
)
meus_localizadores.click()

# Aguarda algum carregamento após o clique, se necessário (exemplo: espera um painel aparecer)
# WebDriverWait(navegador, 20).until(
#     EC.visibility_of_element_located((By.ID, "id_do_painel_ou_elemento_esperado"))
# )

# Localiza o primeiro <td> que contenha "CÍVEL - MINUTAR" no texto
td_civel_minutar = WebDriverWait(navegador, 20).until(
    EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "CÍVEL - MINUTAR")]'))
)

# Encontra o <td> imediatamente a seguir
td_seguinte = td_civel_minutar.find_element(By.XPATH, 'following-sibling::td[1]')

# Dentro desse <td>, localiza o <a> e clica, esperando estar clicável
a_element = WebDriverWait(td_seguinte, 20).until(
    EC.element_to_be_clickable((By.TAG_NAME, 'a'))
)
a_element.click()

# Localiza e clica no label "100 processos por página"
label_100 = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.XPATH, '//label[contains(text(), "100 processos por página")]'))
)
label_100.click()

# Localiza o label com id "lbloptNdiasSituacao" e clica apenas se o checkbox estiver desmarcado
label_ndias_situacao = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.ID, "lbloptNdiasSituacao"))
)
checkbox_ndias = navegador.find_element(By.ID, "optNdiasSituacao")
if not checkbox_ndias.is_selected():
    label_ndias_situacao.click()

# Localiza o botão com id "btnConsultar" e exibe na tela
botao_consultar = navegador.find_element(By.ID, "btnConsultar")
navegador.execute_script("arguments[0].scrollIntoView();", botao_consultar)

# Tenta clicar no botão "Consultar", rolando para garantir visibilidade e tratando possíveis interceptações

try:
    botao_consultar.click()
except ElementClickInterceptedException:
    navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao_consultar)
    time.sleep(1)
    botao_consultar.click()

In [5]:
# Renova a tabela do perfil
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {perfil}")
    conn.commit()

# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()

    cursor.executemany(
        f"INSERT INTO {table_name} (num_processo, dias, tipo) VALUES (?, ?, ?)",
        dados_tabela
    )
    conn.commit()

#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

[['5009747-60.2024.8.21.0009', '12', 'OUTROS PROCEDIMENTOS DE JURISDIÇÃO VOLUNTÁRIA'], ['5009541-70.2021.8.21.0132', '20', 'Guarda'], ['5007601-17.2022.8.21.0009 ', '27', 'PROCEDIMENTO COMUM CÍVEL'], ['5006577-82.2020.8.21.0086', '3', 'Interdição/Curatela'], ['5004573-84.2024.8.21.0069', '7', 'MONITÓRIA'], ['5004547-86.2024.8.21.0069', '12', 'OUTROS PROCEDIMENTOS DE JURISDIÇÃO VOLUNTÁRIA'], ['5004525-28.2024.8.21.0069 ', '45', 'MANDADO DE SEGURANÇA'], ['5004520-40.2023.8.21.0069', '24', 'EXECUÇÃO FISCAL'], ['5004514-33.2023.8.21.0069', '119', 'EXECUÇÃO FISCAL'], ['5004480-58.2023.8.21.0069', '59', 'EXECUÇÃO FISCAL'], ['5004478-88.2023.8.21.0069 ', '20', 'EXECUÇÃO FISCAL'], ['5004467-25.2024.8.21.0069', '34', 'ALIMENTOS - LEI ESPECIAL Nº 5.478/68'], ['5004462-03.2024.8.21.0069 ', '48', 'MONITÓRIA'], ['5004459-48.2024.8.21.0069', '18', 'PROCEDIMENTO COMUM CÍVEL'], ['5004452-90.2023.8.21.0069 ', '20', 'EXECUÇÃO FISCAL'], ['5004434-35.2024.8.21.0069 ', '19', 'EXECUÇÃO FISCAL'], ['5004421-3

In [12]:
# Adiciona mais

# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()

    cursor.executemany(
        f"INSERT INTO {table_name} (num_processo, dias, tipo) VALUES (?, ?, ?)",
        dados_tabela
    )
    conn.commit()

#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

[['5000145-16.2011.8.21.0069', '21', 'PROCEDIMENTO COMUM CÍVEL'], ['5000143-46.2011.8.21.0069 ', '19', 'CUMPRIMENTO DE SENTENÇA CONTRA A FAZENDA PÚBLICA'], ['5000141-37.2015.8.21.0069 ', '10', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL'], ['5000141-27.2021.8.21.0069 ', '21', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL'], ['5000135-93.2016.8.21.0069', '11', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL'], ['5000135-88.2019.8.21.0069 ', '18', 'CUMPRIMENTO DE SENTENÇA'], ['5000135-78.2025.8.21.0069', '18', 'Extinção Consensual de União Estável'], ['5000130-37.2017.8.21.0069', '62', 'PROCEDIMENTO COMUM CÍVEL'], ['5000125-59.2010.8.21.0069 ', '10', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL'], ['5000123-55.2011.8.21.0069 ', '12', 'CUMPRIMENTO DE SENTENÇA CONTRA A FAZENDA PÚBLICA'], ['5000119-47.2013.8.21.0069', '3', 'CUMPRIMENTO DE SENTENÇA'], ['5000119-03.2020.8.21.0069', '35', 'ARROLAMENTO SUMÁRIO'], ['5000117-82.2010.8.21.0069 ', '12', 'CUMPRIMENTO DE SENTENÇA'], ['5000117-62.2022.8.21.0069 ', '19', 'PROCEDIMENTO COMUM CÍVEL']

In [46]:
#Gera lista de pendentes, do tipo de processo definido

with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(
        f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}' AND pet IS NULL", conn
    )
df_pendentes = df_pendentes["num_processo"].tolist()

print(f"Total de processos pendentes: {len(df_pendentes)}")


Total de processos pendentes: 1


In [47]:
#EXECUTA! Pega as últimas petições
def captura_peticoes(navegador, processo):
    
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        evento_id, texto_peticao = pega_ultima_peticao(navegador)

        print(f"========== TEXTO DA PETIÇÃO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()      
        
        


for processo in df_pendentes:
    captura_peticoes(navegador, processo)

Página do processo 5002946-79.2023.8.21.0069 carregada com sucesso.
========== TEXTO DA PETIÇÃO ==========
1
MUNICÍPIO DE SARANDI
ESTADO DO RIO GRANDE DO SUL
ASSESSORIA JURÍDICA MUNICIPAL
A(O) EXCELENTÍSSIMO(A) SENHOR(A) DOUTOR(A) JUIZ(A) DE DIREITO DA VARA
JUDICIAL DA COMARCA DE SARANDI/RS.
MUNICÍPIO DE SARANDI – RS, nos presentes autos, por sua
procuradora signatária, vêm respeitosamente perante Vossa Excelência, dizer e
requerer o que segue:
Considerando o teor da Resolução nº 547 do CNJ que institui
medidas de tratamento racional e eficiente na tramitação das execuções fiscais
à partir do Tema 1184 do STF, o exequente REQUER a suspensão da execução
pelo período de 180 dias, nos moldes do art. 313, II, § 4º para fins de atendimento
ao determinado na referida Resolução, bem como protesto dos créditos
tributários.
Termos em que pede deferimento.
Sarandi/RS, data do protocolo.

 Karine Fabia Davoglio Mazzetti Barella
 OAB/RS 86.438
 Assessora Jurídica
 


In [5]:
# Gera resumo das petições do db

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT num_processo, pet FROM {perfil} WHERE pet IS NOT NULL AND resumo IS NULL")
    processos = cursor.fetchall()

    for num_processo, texto_pet in processos:
        print(f"Processo: {num_processo}")
        resumo = ollama_resumo(texto_pet)
        print(f"Resumo: {resumo}")
        cursor.execute(
            f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",
            (resumo, num_processo)
        )
        conn.commit()
    

Processo: 5004514-33.2023.8.21.0069
========== Iniciando resumo com LLM... ==========
Resumo: Solicita-se a suspensão do processo devido ao cumprimento do acordo de parcelamento do débito.
Processo: 5004480-58.2023.8.21.0069
========== Iniciando resumo com LLM... ==========
Resumo: Solicita-se a desconsideração do valor irrisório e o prosseguimento do processo de execução fiscal.

Processo: 5004478-88.2023.8.21.0069 
========== Iniciando resumo com LLM... ==========
Resumo: A municipalidade solicita o cancelamento do parcelamento e a busca de valores em contas bancárias, por meio do sistema Sisbajud.
Processo: 5004452-90.2023.8.21.0069 
========== Iniciando resumo com LLM... ==========
Resumo: O município requer a citação do executado por meio de ligação telefônica ou mensagem de WhatsApp, com comprovação inequívoca do recebimento.
Processo: 5004408-71.2023.8.21.0069
========== Iniciando resumo com LLM... ==========
Resumo: O Município solicita um prazo de 180 dias para realizar o prot

In [33]:
#EXECUTA! Pega e resume as últimas petições
def analisa_processo(navegador, processo):
    
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        evento_id, texto_peticao = pega_ultima_peticao(navegador)

        print(f"========== TEXTO DA PETIÇÃO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()
        
        resumo_ollama = ollama_resumo(texto_peticao)
        print(f"========== RESUMO DA PETIÇÃO ==========")
        print(resumo_ollama)

        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",    
                (resumo_ollama, processo)
            )
            conn.commit()
        #verifica os tipos de pedidos conhecidos
        with sqlite3.connect("movimentos.db") as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT id, resumo FROM pedidos")
            tipos_pedidos = cursor.fetchall()
            tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])
        
        resultado = verifica_tipos_de_pedidos(resumo_ollama, tipos_pedidos)
        print(resultado)
        resultado_limpo = re.sub(r'[\r\n]+', ' ', resultado).strip()
        resultado = re.sub(r'[^\w\s.,;:!?-]', '', resultado_limpo)       
        
        # Limpa o resultado de retornos e caracteres especiais




for processo in df_pendentes:
    analisa_processo(navegador, processo)



Página do processo 5004520-40.2023.8.21.0069 carregada com sucesso.
========== TEXTO DA PETIÇÃO ==========
ESTADO DO RIO GRANDE DO SUL
MUNICÍPIO DE SARANDI
EXCELENTÍSSIMA SENHORA DOUTORA JUÍZA DE DIREITO DA VARA
JUDICIAL DA COMARCA DE SARANDI-RS
Processo N° 5004520-40.2023.8.21.0069
MUNICÍPIO DE SARANDI, por sua procuradora, no feito supra
epigrafado, que move em face de RONALDO GOIS VARGAS, vem, respeitosamente, à
presença de Vossa Excelência, dizer e requerer o que segue:
A fim de dar seguimento ao processo, considerando o estorno do
parcelamento, REQUER que Vossa Excelência requisite junto ao SISBAJUD a penhora
de valores em nome de RONALDO GOIS VARGAS, inscrito no CPF nº
006.290.910-05, determinando o bloqueio de valores suficientes para o pagamento do
débito e a reiteração automática de ordem de bloqueio, de forma contínua por 30
dias, até o bloqueio do valor necessário para a o seu total cumprimento.
É importante consignar que o pedido está em consonância com o
entendimento juris

PyperclipWindowsException: Error calling OpenClipboard ([WinError 0] A operação foi concluída com êxito.)

In [ ]:
#Lembretes

data_atual = datetime.now().strftime("%d/%m/%Y")
texto_lembrete = "Solicita-se a desconsideração do valor de execução irrisório e o prosseguimento do processo."
lembrete = f"{data_atual} - {texto_lembrete}"

def insere_lembrete(navegador, texto):
    # Localiza o campo de lembrete e insere o texto    
    navegador.switch_to.default_content()
    novo_btn = navegador.find_element(By.LINK_TEXT, "Novo")

    time.sleep(0.5)
    novo_btn.click()
    navegador.switch_to.frame(1)
    navegador.find_element(By.ID, "txaDescricao").click()
    navegador.find_element(By.ID, "txaDescricao").send_keys(texto)
    navegador.find_element(By.CSS_SELECTOR, "td:nth-child(2) > .infraRadio").click()
    navegador.find_element(By.CSS_SELECTOR, "#divInfraBarraComandosInferior > #sbmSalvar").click()
    navegador.switch_to.default_content()
    time.sleep(2)

# Insere o lembrete no processo
insere_lembrete(navegador, lembrete)

In [7]:
# Mostra os tipos de pedidos conhecidos'
with sqlite3.connect("movimentos.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT id, resumo FROM pedidos")
    tipos_pedidos = cursor.fetchall()
    tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])
    print(tipos_pedidos)

ID: 1, Resumo: Pedido de desistência do processo
ID: 2, Resumo: Informação de que a parte está ciente.
ID: 3, Resumo: Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança de dívida tributária.
ID: 4, Resumo: Pedido de desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 5, Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal de baixo valor.
ID: 6, Resumo: Solicita-se a desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 7, Resumo: Solicita-se a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas da executada.


In [10]:

eproc.apaga_ultimo_lembrete(navegador)

In [33]:
print(verifica_tipos_de_pedidos(resumo, tipos_pedidos))

========== Verificando se é um caso de uso conhecido... ==========
Sim
4



In [ ]:
#EXECUTA! Pega e resume as últimas petições
for processo in df_pendentes:
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        texto_peticao = pega_ultima_peticao(navegador)

        print("========== RESUMO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()
        
        resumo_ollama = ollama_resumo(texto_peticao)
        print("========== RESUMO ==========")
        print(resumo_ollama)
        print("============================")

        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",    
                (resumo_ollama, processo)
            )
            conn.commit()



Página do processo 5004481-43.2023.8.21.0069 carregada com sucesso.
========== RESUMO ==========

ESTADO DO RIO GRANDE DO SUL
PREFEITURA MUNICIPAL DE BARRA FUNDA
Av. 24 de Março, 735 – Centro – Fone (54) 99655-8503 – Cep 99.585-000 – Barra Funda - RS 1
AO JUÍZO DA VARA JUDICIAL DA COMARCA DE SARANDI/RS.
EXECUÇÃO FISCAL Nº 5004481-43.2023.8.21.0069
MUNICÍPIO DE BARRA FUNDA, pessoa Jurídica de Direito Público interno, por
meio da assessora jurídica que subscreve, vem respeitosamente perante Vossa
Excelência, nos autos do processo que move em face de SUZANA ANDRADE, requerer
o que segue:
Considerando o despacho proferido sobre a aplicação do Tema 1184 de
Repercussão Geral do STF e da Resolução n. 547 do CNJ, que dispõem sobre a
extinção de execuções fiscais de baixo valor, e tendo em vista que o protesto do
título é uma condição prevista para o ajuizamento da execução fiscal e que a
Fazenda Pública não teve a oportunidade de realizá-lo, requer-se a concessão de
prazo de 180 (cento e oiten

InvalidSessionIdException: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x0x7ff7221d6f75+76917]
	GetHandleVerifier [0x0x7ff7221d6fd0+77008]
	(No symbol) [0x0x7ff721f89c1c]
	(No symbol) [0x0x7ff721fd055f]
	(No symbol) [0x0x7ff722008332]
	(No symbol) [0x0x7ff722002e53]
	(No symbol) [0x0x7ff722001f19]
	(No symbol) [0x0x7ff721f54b05]
	GetHandleVerifier [0x0x7ff7224ad2ad+3051437]
	GetHandleVerifier [0x0x7ff7224a7903+3028483]
	GetHandleVerifier [0x0x7ff7224c589d+3151261]
	GetHandleVerifier [0x0x7ff7221f183e+185662]
	GetHandleVerifier [0x0x7ff7221f96ff+218111]
	(No symbol) [0x0x7ff721f53b00]
	GetHandleVerifier [0x0x7ff7225c5f18+4201496]
	BaseThreadInitThunk [0x0x7ffda032e8d7+23]
	RtlUserThreadStart [0x0x7ffda0a7c34c+44]


In [7]:
#Cria um digesto:
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"SELECT num_processo, resumo FROM {perfil} WHERE resumo IS NOT NULL")
    resultados = cursor.fetchall()
    for num_processo, resumo in resultados:
        print(f"Processo: {num_processo}")
        print(f"Resumo: {resumo}")
        print("-" * 40)

Processo: 5004481-43.2023.8.21.0069
Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal de baixo valor, em conformidade com as determinações do STF e do CNJ.
----------------------------------------
Processo: 5004480-58.2023.8.21.0069
Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal e ao cumprimento do devido processo legal.
----------------------------------------
Processo: 5004479-73.2023.8.21.0069
Resumo: Solicita-se prorrogação do prazo para realizar o protesto do título, sob a alegação de que o prazo inicial é insuficiente para a devida condução do processo de extinção da execução fiscal.
----------------------------------------
Processo: 5004411-26.2023.8.21.0069 
Resumo: O Município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal.
----------------------------------------
Processo: 5004